# 🔍 Critical Sanity Checks for MultiClassDataset

Run these tests BEFORE starting training to verify:
1. Tensor shapes and data types
2. Visual alignment (CRS correctness)
3. Multi-class distribution

**Stop and debug if any test fails.**

## Setup

In [ ]:
import sys
from pathlib import Path

# Add project root to path
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

import torch
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter

from src.datasets.multiclass_dataset import MultiClassDataset, get_train_transform, get_val_transform

print("✓ Imports successful")

## TEST 1️⃣: Shape and Data Type Validation

**Expected Results:**
- Image shape: `torch.Size([3, 512, 512])`
- Mask shape: `torch.Size([512, 512])`
- Mask dtype: `torch.int64`
- Mask values: subset of `{0, 1, 2, 3, 4}`

In [ ]:
# Instantiate dataset
print("Creating dataset...")
dataset = MultiClassDataset(
    data_root="data/Raz/PB_training_dataSet_shp_file",
    split="train",
    transform=get_train_transform(512),
    patch_size=512,
    patches_per_image=50,
    positive_ratio_threshold=0.03,
    positive_sampling_prob=0.7,
    debug_sampling=False,
)
print(f"✓ Dataset created with {len(dataset)} samples\n")

# Get first sample
print("Loading first sample...")
img, mask = dataset[0]

print("\n" + "="*60)
print("SHAPE TEST RESULTS")
print("="*60)
print(f"Image shape:  {img.shape}")
print(f"Image dtype:  {img.dtype}")
print(f"Image range:  [{img.min():.4f}, {img.max():.4f}]")
print()
print(f"Mask shape:   {mask.shape}")
print(f"Mask dtype:   {mask.dtype}")
print(f"Mask values:  {mask.unique().tolist()}")
print(f"Mask range:   [{mask.min()}, {mask.max()}]")
print("="*60)

# Validation checks
print("\nVALIDATION CHECKS:")
checks_passed = 0
checks_total = 4

if img.shape == torch.Size([3, 512, 512]):
    print("  ✓ Image shape correct")
    checks_passed += 1
else:
    print(f"  ✗ Image shape incorrect! Got {img.shape}, expected (3, 512, 512)")

if mask.shape == torch.Size([512, 512]):
    print("  ✓ Mask shape correct")
    checks_passed += 1
else:
    print(f"  ✗ Mask shape incorrect! Got {mask.shape}, expected (512, 512)")

if mask.dtype == torch.int64:
    print("  ✓ Mask dtype correct (int64/long)")
    checks_passed += 1
else:
    print(f"  ✗ Mask dtype incorrect! Got {mask.dtype}, expected torch.int64")
    print("  ⚠️  STOP: CrossEntropyLoss requires long dtype")

unique_vals = set(mask.unique().tolist())
valid_vals = {0, 1, 2, 3, 4}
if unique_vals.issubset(valid_vals):
    print(f"  ✓ Mask values valid: {unique_vals}")
    checks_passed += 1
else:
    print(f"  ✗ Mask contains invalid values! Got {unique_vals}, expected subset of {valid_vals}")
    print("  ⚠️  STOP: Invalid class IDs detected")

print(f"\n✓ Passed {checks_passed}/{checks_total} checks")


## TEST 2️⃣: Visual Overlay Test (CRS Alignment Verification)

**What to look for:**
- Mask should align with roads, buildings, and geographic features in the image
- If mask looks random or spatially shifted → CRS alignment failed
- If mask is mostly black → rasterization may have failed

In [ ]:
# Create visualization
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Plot 1: Image
img_display = img.permute(1, 2, 0).numpy()
# Normalize to [0, 1] for display
img_display = (img_display - img_display.min()) / (img_display.max() - img_display.min() + 1e-6)
axes[0].imshow(img_display)
axes[0].set_title("Original Image")
axes[0].axis("off")

# Plot 2: Mask (grayscale)
mask_display = mask.numpy().astype(np.uint8)
axes[1].imshow(mask_display, cmap="tab10")
axes[1].set_title("Segmentation Mask\n(Classes 0-4)")
axes[1].axis("off")
cbar1 = plt.colorbar(axes[1].images[0], ax=axes[1], label="Class ID")

# Plot 3: Overlay (50% transparency)
overlay = img_display.copy()
mask_colored = np.zeros((*mask_display.shape, 3))

# Color map for classes
colors = {
    0: [0, 0, 0],        # background - black (transparent)
    1: [1, 0, 0],        # road - red
    2: [0, 1, 0],        # railway - green
    3: [0, 0, 1],        # bridge - blue
    4: [1, 1, 0],        # built_up - yellow
}

for class_id, color in colors.items():
    mask_colored[mask_display == class_id] = color

overlay_alpha = 0.4
blended = (1 - overlay_alpha) * img_display + overlay_alpha * mask_colored
axes[2].imshow(blended)
axes[2].set_title("Overlay\n(Image + Mask)")
axes[2].axis("off")

plt.tight_layout()
plt.savefig("/tmp/dataset_sanity_visual.png", dpi=100, bbox_inches="tight")
plt.show()

print("✓ Visual saved to /tmp/dataset_sanity_visual.png")
print("\nVISUAL INSPECTION CHECKLIST:")
print("  □ Does the mask show spatial structure (not random)?")
print("  □ Do colored regions align with features in the image?")
print("  □ Is the mask mostly zeros (background) with some non-zero regions?")
print("\n  If all ☑, CRS alignment is likely correct.")
print("  If any ☐, CRS may still need fixing.")

## TEST 3️⃣: Multi-Class Distribution Check

**Expected Results:**
- All classes should appear with non-zero counts
- Class 0 (background) should be most common
- Classes 1–4 (features) should have lower but non-zero counts

**If all counts are near zero** → rasterization failing silently

In [ ]:
print("Sampling 50 patches to analyze class distribution...")
print("(This may take 1-2 minutes)\n")

class_counts = Counter()
patch_counts = 0

for i in range(50):
    try:
        _, mask = dataset[i]
        unique_classes = mask.unique().tolist()
        for class_id in unique_classes:
            class_counts[class_id] += (mask == class_id).sum().item()
        patch_counts += 1
        if (i + 1) % 10 == 0:
            print(f"  Processed {i+1}/50 patches")
    except Exception as e:
        print(f"  ✗ Error loading patch {i}: {e}")
        continue

print(f"\n✓ Successfully processed {patch_counts}/50 patches\n")

print("="*60)
print("CLASS DISTRIBUTION (pixel counts)")
print("="*60)

class_names = {
    0: "Background",
    1: "Road",
    2: "Railway",
    3: "Bridge",
    4: "Built-Up Area",
}

total_pixels = sum(class_counts.values())

for class_id in sorted(class_counts.keys()):
    count = class_counts[class_id]
    percentage = 100 * count / total_pixels if total_pixels > 0 else 0
    bar_length = int(percentage / 2)
    bar = "█" * bar_length + "░" * (50 - bar_length)
    print(f"Class {class_id} ({class_names.get(class_id, 'Unknown'):15s}): {count:12d} ({percentage:6.2f}%) {bar}")

print("="*60)

# Validation
print("\nDISTRIBUTION VALIDATION:")
checks_passed = 0
checks_total = 3

# Check 1: All classes present
expected_classes = {0, 1, 2, 3, 4}
present_classes = set(class_counts.keys())

if 0 in present_classes:
    print("  ✓ Background (class 0) present")
    checks_passed += 1
else:
    print("  ✗ Background (class 0) missing!")

feature_classes = {1, 2, 3, 4} & present_classes
if len(feature_classes) > 0:
    print(f"  ✓ Feature classes present: {sorted(feature_classes)}")
    checks_passed += 1
else:
    print("  ✗ No feature classes found! Rasterization may be failing.")

# Check 2: Reasonable distribution
if class_counts[0] > total_pixels * 0.5:  # Background should be >50%
    print("  ✓ Background dominates (>50%) - reasonable for sparse features")
    checks_passed += 1
else:
    print(f"  ✗ Background only {100*class_counts[0]/total_pixels:.1f}% - may indicate data issue")

print(f"\n✓ Passed {checks_passed}/{checks_total} checks")

## ✅ Final Verdict

**If all three tests passed:**
- ✅ Dataset is ready for training
- ✅ Tensor shapes are correct for PyTorch
- ✅ CRS alignment is working
- ✅ Rasterization is functioning

**If any test failed:**
- 🛑 DEBUG before training
- Check error messages above
- Verify SHP file paths
- Confirm CRS in both rasters and vectors

In [ ]:
print("\n" + "="*60)
print("FINAL STATUS")
print("="*60)
print("All sanity checks completed.")
print("\nIf all tests above show ✓:")
print("  → Ready to start training with: python train.py")
print("\nIf any test shows ✗:")
print("  → Review error messages and debug dataset configuration")
print("  → Do NOT proceed to training")
print("="*60)